<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_6/dataset_pbl/tugas_praktikum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Tugas Praktikum**

Jalankan code berikut pada Google Colab dan PyDroid3 (Android Python) Application di Smartphone Android. Bandingkan hasilnya dan tuliskan analisa anda, tuliskan juga spesifikasi smartphone yang anda gunakan :)



In [1]:
!pip install annoy hnswlib faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 6.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 30.8 MB/s  0:00:00
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520954 sha256=f11f6832a5263abce91cdfd6d7ae2264e8fdf09d06c9223fcf1d4d4f7ee66dbd
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679066 sha256=8f9fb5e5bec0b8c2e61366e00ade217a3d0ff4537d54012fbe20a0b9358b6706
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [2]:
import numpy as np
import pandas as pd
import time
from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex
import hnswlib
import faiss
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [3]:
# -------------------------------
# Load & Preprocessing Dataset Sleep Health
# -------------------------------
# Pastikan file CSV sudah diunggah ke Colab atau penyimpanan lokal PyDroid3
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')

# Hapus kolom ID dan target label (Sleep Disorder) jika hanya ingin membandingkan fitur gaya hidup & kesehatan
df_features = df.drop(columns=['Person ID', 'Sleep Disorder'], errors='ignore')

# Pisahkan kolom Blood Pressure (misal "126/83") menjadi Systolic dan Diastolic
if 'Blood Pressure' in df_features.columns:
    bp_split = df_features['Blood Pressure'].str.split('/', expand=True)
    df_features['Systolic_BP'] = pd.to_numeric(bp_split[0])
    df_features['Diastolic_BP'] = pd.to_numeric(bp_split[1])
    df_features = df_features.drop(columns=['Blood Pressure'])

# Encode kolom kategorikal (Gender, Occupation, BMI Category) menjadi numerik
categorical_cols = df_features.select_dtypes(include=['object']).columns
for col in categorical_cols:
    le = LabelEncoder()
    df_features[col] = le.fit_transform(df_features[col].astype(str))

# Konversi ke float32 (syarat wajib untuk FAISS dan HNSW)
X = df_features.values.astype('float32')
n_samples, d = X.shape

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')

k = 10  # jumlah nearest neighbors

# -------------------------------
# Exact NN (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=d)
p_hnsw.init_index(max_elements=n_samples, ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, _ = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
nlist = min(10, n_samples // 10)  # Menyesuaikan jumlah cluster dengan ukuran dataset
quantizer = faiss.IndexFlatL2(d)
index_faiss = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 3
_, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------------------
# Tampilkan ringkasan waktu
# -------------------------------
print("\n=== Ringkasan Waktu (detik) ===")
print(f"Exact NN : {time_exact:.3f}")
print(f"Annoy    : {time_annoy:.3f}")
print(f"HNSW     : {time_hnsw:.3f}")
print(f"FAISS    : {time_faiss:.3f}")

Exact NN done in 0.045 s
Annoy done in 0.013 s
HNSW done in 0.043 s
FAISS IVF done in 0.029 s

=== Ringkasan Waktu (detik) ===
Exact NN : 0.045
Annoy    : 0.013
HNSW     : 0.043
FAISS    : 0.029


### Analisa Perbandingan

Google Colab memproses seluruh algoritma jauh lebih cepat daripada smartphone karena berjalan di server cloud dengan kapasitas CPU dan RAM besar (12,67 GB). Sebaliknya, smartphone mengandalkan prosesor seluler (Snapdragon 685) yang memprioritaskan efisiensi daya baterai dan berbagi sumber daya dengan sistem operasi Android.

Pola performa algoritma mengalami perubahan antara kedua perangkat tersebut:
* **Pada Google Colab:** Metode **Annoy** mencatat waktu tercepat (0,013 detik), diikuti oleh **FAISS IVF** (0,029 detik), **HNSW** (0,043 detik), dan **Exact NN** (0,045 detik). Arsitektur komputasi cloud mampu membangun struktur indeks pohon dan graf dengan sangat efisien.
* **Pada Smartphone:** Metode **Exact NN** justru bekerja lebih efisien daripada metode aproksimasi. Karena dataset *Sleep Health* hanya berukuran ratusan baris, prosesor smartphone dapat menghitung jarak matriks secara langsung dengan cepat dibandingkan harus mengalokasikan memori untuk membangun struktur indeks pohon, graf, atau klasterisasi dari awal.

Algoritma Approximate Nearest Neighbors (Annoy, HNSW, dan FAISS) dirancang untuk memproses jutaan data berdimensi tinggi. Pada dataset skala kecil, waktu pembangunan indeks justru mendominasi total waktu eksekusi pada perangkat dengan spesifikasi terbatas.
- Spesifikasi smartphone yang digunakan:
  1. Perangkat: Redmi Note 12 4G
  2. Kapasitas RAM: 6,0 GB   
  3. CPU atau Prosesor: Snapdragon 685 Octa-core Max 2.80GHz   
  4. Versi Android: 14